# Model Training

In [1]:
# Import required libraries
import torch
import numpy as np
import os
from pathlib import Path
import matplotlib.pyplot as plt

# Import our modules
import data
import model
from training import train_model

## Training Configuration

In [ ]:
# Training hyperparameters
MAX_EPOCHS = 100
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 1e-5
PATIENCE = 7  # Early stopping patience
BATCH_SIZE = 32
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Training frequency
FREQUENCY = 1000  # Hz

print(f"Training on device: {DEVICE}")
print(f"Sampling frequency: {FREQUENCY}Hz")

## Model Training

Train the model and save the best checkpoint based on validation loss.

In [ ]:
# Data directory
data_dir = f"dataset/1000Hz"

# Check if data exists
if not os.path.exists(data_dir):
    raise FileNotFoundError(f"Data directory {data_dir} does not exist")

# Create dataset and dataloaders
dataset = data.ImpactDataset(data_dir=data_dir, force_bounds=(2, 3))
loaders = data.ImpactDataloader(dataset=dataset, batch_size=BATCH_SIZE)

print(f"\n{'='*70}")
print(f"Training model for {FREQUENCY}Hz data")
print(f"Dataset: {len(dataset)} samples")
print(f"  Train: {len(loaders['train'].dataset)}")
print(f"  Val: {len(loaders['val'].dataset)}")
print(f"  Test: {len(loaders['test'].dataset)}")
print(f"{'='*70}\n")

# Create and train model
model_instance = model.Attention1DConv()
history, trained_model = train_model(
    model=model_instance,
    loaders=loaders,
    max_epochs=MAX_EPOCHS,
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    patience=PATIENCE,
    device=DEVICE
)

# Save best model
save_dir = Path("../results/models")
save_dir.mkdir(parents=True, exist_ok=True)

best_val_loss = min(history['val_loss'])
best_epoch = history['val_loss'].index(best_val_loss) + 1

model_path = save_dir / f"model_{FREQUENCY}Hz_epoch{best_epoch:03d}_valloss{best_val_loss:.6f}.pt"
torch.save({
    'model_state_dict': trained_model.state_dict(),
    'frequency': FREQUENCY,
    'best_epoch': best_epoch,
    'best_val_loss': best_val_loss,
    'history': history
}, model_path)

print(f"\n{'='*70}")
print(f"Model saved: {model_path.name}")
print(f"Best epoch: {best_epoch}")
print(f"Best validation loss: {best_val_loss:.6f}")
print(f"{'='*70}")

## Results Visualization

In [ ]:
# Plot training curves
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

metrics = [
    ('train_loss', 'val_loss', 'Total Loss'),
    ('train_loc', 'val_loc', 'Location Loss (MSE)'),
    ('train_force', 'val_force', 'Force Loss (CE)'),
    ('train_acc', 'val_acc', 'Force Classification Accuracy')
]

for i, (train_key, val_key, title) in enumerate(metrics):
    ax = axes[i]
    epochs = range(1, len(history[train_key]) + 1)

    ax.plot(epochs, history[train_key], '--', alpha=0.7, linewidth=1.5, label='Train')
    ax.plot(epochs, history[val_key], '-', linewidth=2, label='Validation')

    # Mark best epoch
    best_idx = history['val_loss'].index(min(history['val_loss']))
    ax.axvline(best_idx + 1, color='red', linestyle=':', alpha=0.5, label='Best Epoch')

    ax.set_xlabel('Epoch', fontsize=11)
    ax.set_ylabel(title, fontsize=11)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Training Summary

In [ ]:
# Training summary
print("Training Summary")
print("=" * 70)
print(f"Sampling frequency: {FREQUENCY}Hz")
print(f"Dataset size: {len(dataset)}")
print(f"Training epochs: {best_epoch}")
print(f"\nBest validation metrics (epoch {best_epoch}):")
print(f"  Loss:      {history['val_loss'][best_epoch-1]:.6f}")
print(f"  Loc MSE:   {history['val_loc'][best_epoch-1]:.6f}")
print(f"  Force CE:  {history['val_force'][best_epoch-1]:.6f}")
print(f"  Force Acc: {history['val_acc'][best_epoch-1]:.4f}")
print(f"\nFinal training metrics (epoch {len(history['train_loss'])}):")
print(f"  Loss:      {history['train_loss'][-1]:.6f}")
print(f"  Loc MSE:   {history['train_loc'][-1]:.6f}")
print(f"  Force CE:  {history['train_force'][-1]:.6f}")
print(f"  Force Acc: {history['train_acc'][-1]:.4f}")
print("=" * 70)